In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
# Import SparkSession to start Spark
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("Task2_ML").getOrCreate()

print("Spark Ready")

Spark Ready


In [ ]:
# Import pandas to load dataset from URL
import pandas as pd

# Dataset URL
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

# Load dataset into pandas DataFrame
pdf = pd.read_csv(url)
# Save dataset locally as CSV
pdf.to_csv("tips.csv", index=False)

In [ ]:
# Load CSV into PySpark DataFrame
df = spark.read.csv("tips.csv", header=True, inferSchema=True)

# Display Dataset
df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|
|     18.43| 3.0|  Male|    No|Sun|Dinner|   4|
|     14.83|3.02|Female|    No|Sun|Dinner|   2|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     10.33|1.67|Female|    No|Sun|Dinner|   3|
|     16.29|3.71|  Male|    No|Sun|Dinne

In [ ]:
# Import VectorAssembler (used to prepare features for ML model)
from pyspark.ml.feature import VectorAssembler

# Convert 'total_bill' column into feature vector
assembler = VectorAssembler(inputCols=["total_bill"], outputCol="features")

# Transform dataset to include 'features' column
data = assembler.transform(df)

# Display features and target variable
data.select("features", "tip").show()

+--------+----+
|features| tip|
+--------+----+
| [16.99]|1.01|
| [10.34]|1.66|
| [21.01]| 3.5|
| [23.68]|3.31|
| [24.59]|3.61|
| [25.29]|4.71|
|  [8.77]| 2.0|
| [26.88]|3.12|
| [15.04]|1.96|
| [14.78]|3.23|
| [10.27]|1.71|
| [35.26]| 5.0|
| [15.42]|1.57|
| [18.43]| 3.0|
| [14.83]|3.02|
| [21.58]|3.92|
| [10.33]|1.67|
| [16.29]|3.71|
| [16.97]| 3.5|
| [20.65]|3.35|
+--------+----+
only showing top 20 rows


In [ ]:
# Import Linear Regression model
from pyspark.ml.regression import LinearRegression

# Create Linear Regression model
# featuresCol = input column
# labelCol = output column (what we want to predict)
lr = LinearRegression(featuresCol="features", labelCol="tip")

# Train model using dataset
model = lr.fit(data)

In [ ]:
# Make predictions using trained model
predictions = model.transform(data)

# Show actual vs predicted values
predictions.select("total_bill", "tip", "prediction").show()

+----------+----+------------------+
|total_bill| tip|        prediction|
+----------+----+------------------+
|     16.99|1.01|2.7046361639148353|
|     10.34|1.66| 2.006223123308884|
|     21.01| 3.5|3.1268347237999374|
|     23.68|3.31|3.4072501852161614|
|     24.59|3.61| 3.502822496035923|
|     25.29|4.71| 3.576339658204971|
|      8.77| 2.0|1.8413346310154486|
|     26.88|3.12| 3.743328640846093|
|     15.04|1.96| 2.499838355015346|
|     14.78|3.23| 2.472531980495414|
|     10.27|1.71| 1.998871407091979|
|     35.26| 5.0| 4.623434096526977|
|     15.42|1.57|2.5397476716214005|
|     18.43| 3.0|2.8558714689483047|
|     14.83|3.02|2.4777832063646317|
|     21.58|3.92|3.1866986987090185|
|     10.33|1.67|  2.00517287813504|
|     16.29|3.71| 2.631119001745788|
|     16.97| 3.5|2.7025356735671484|
|     20.65|3.35|  3.08902589754157|
+----------+----+------------------+
only showing top 20 rows


In [ ]:
# Print model evaluation details
print("Model Coefficients:", model.coefficients)
print("Model Intercept:", model.intercept)

Model Coefficients: [0.10502451738435366]
Model Intercept: 0.920269613554667
